In [1]:
import sys
print(sys.executable)

/mnt/d/DEV/llm-zoomcamp2026/02-vector-search/code/.venv/bin/python


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
hf_token=os.getenv("HF_TOKEN")
if hf_token is None:
    raise ValueError("HF_TOKEN environment variable not set")
else:
    print("HF_TOKEN is set correctly")


HF_TOKEN is set correctly


In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("./all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
q2 = "Can I enroll in the course after it has started?"
v2 = model.encode(q2)
q3 = "What is the refund policy for the course?"
v3 = model.encode(q3)
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
print(f"Length of v1: {len(v1)}")
print(f"Length of v2: {len(v2)}")
print(f"Length of v3: {len(v3)}")
print(f"Length of dv: {len(dv)}")

Length of v1: 384
Length of v2: 384
Length of v3: 384
Length of dv: 384


In [6]:
v1.dot(dv)

np.float32(0.32332397)

In [7]:
q5 = "How to install Docker on Windows?"
v5 = model.encode(q5)

In [8]:
v5.dot(dv)

np.float32(0.019730486)

In [11]:
from ingest import load_faq_data

documents = load_faq_data()

print(f"Number of documents: {len(documents)}")

Number of documents: 1350


In [12]:
documents[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 'doc_id': '9e508f2212'}

In [13]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [14]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [15]:
import numpy as np
X = np.array(vectors)
X.shape

(1350, 384)

In [16]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [17]:
scores = X.dot(v_query)

In [20]:
idx = np.argmax(scores)
idx, scores[idx]


(np.int64(2), np.float32(0.7629412))

In [21]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [23]:
top5 = np.argsort(scores)[-5:]

top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [24]:
scores[top5]

array([0.7629412 , 0.7579371 , 0.7192131 , 0.65363115, 0.56009996],
      dtype=float32)

In [25]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629412
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions',

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [27]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [28]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [29]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [47]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
if os.getenv("ANTHROPIC_API_KEY") is None:
    raise ValueError("ANTHROPIC_API_KEY environment variable not set")
else:
    print("ANTHROPIC_API_KEY is set correctly")
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


ANTHROPIC_API_KEY is set correctly


In [48]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [49]:
from rag_helper_anthropic import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=anthropic_client,
)

In [50]:
query = "I just found out about the program, can I still sign up?"
answer, usage = assistant.rag(query)
print(answer)
print(f"Input tokens: {usage.input_tokens}")

## Yes, You Can Still Join! 🎉

Based on the course information, **yes, you can still sign up**. However, there is an important condition to keep in mind:

> **If you want to receive a certificate**, you need to make sure to **submit your project while submissions are still being accepted**.

### Key Points to Remember:
- **Homework is not mandatory**, but it is recommended to reinforce concepts
- To earn a certificate, you need to **pass the Capstone Project**
- Homework points count toward your **leaderboard ranking**

So the sooner you join and get started, the better your chances of completing everything in time! Good luck! 🚀
Input tokens: 559


In [51]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [52]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=anthropic_client,
)

In [53]:
vector_assistant.rag("the program has already begun, can I still sign up?")

("Based on the context provided, **yes, you can still join!** However, there is an important condition to keep in mind:\n\n- If you want to **receive a certificate**, you need to make sure you submit your project **while submissions are still being accepted**.\n\nAlso, note that you don't necessarily need to formally register — you can simply **start learning and submitting homework** while the submission forms are still open.",
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=481, output_tokens=91, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [54]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [55]:
vs_index.fit(vectors, documents)

In [58]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5)

In [59]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [60]:
vs_index.close()